# Noise Suppression

Build a spectral noise suppressor, then find where it helps, where it hurts, and where it cannot help at all. Predict each result before running the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

clean = main.synthetic_speech()
noisy, noise = main.mix_at_snr(clean, main.make_noise('fan', len(clean)), 5.0)
print('input SNR', round(main.snr_db(clean, noisy), 1), 'dB')

## Can we track the noise while the caller is speaking?

In [ ]:
est = main.minimum_statistics(np.abs(main.stft(noisy)) ** 2)
truth = np.abs(main.stft(noise)) ** 2
print('tracked / true noise level:', round(10 * np.log10(est[100:].mean() / truth[100:].mean()), 2), 'dB')

## Three suppressors

Compare output SNR with the amount of musical noise left in the pauses.

In [ ]:
pauses = main.speech_absent_frames(clean)
for name, method, kw in (('subtraction, no floor', 'subtraction', {'over': 1.0, 'floor': 0.0}),
                         ('subtraction, floor 0.1', 'subtraction', {'over': 2.0, 'floor': 0.1}),
                         ('Wiener + DD', 'wiener', {})):
    out, gain = main.enhance(noisy, method, return_gain=True, **kw)
    print(f"{name:<23} SNR {main.snr_db(clean, out):5.1f} dB   musical {100 * main.musical_noise(noise, gain, pauses):4.1f}%")

## Over-suppression

Push the noise estimate up. When does the output SNR start to fall?

In [ ]:
spec = main.stft(noisy)
power = np.abs(spec) ** 2
psd = main.minimum_statistics(power)
for over in (0.5, 1, 2, 4, 8, 16):
    gain = main.wiener_dd_gain(power, psd, over=over, floor=0.01)
    d, r = main.distortion_and_reduction(clean, noise, gain)
    out = main.istft(gain * spec, len(noisy))
    print(f"over {over:>4}x  noise -{r:4.1f} dB  distortion {d:6.1f} dB  output SNR {main.snr_db(clean, out):5.1f} dB")

## Noise that looks like speech

Is the problem the noise *estimate* or the gain rule? Give the suppressor the true noise power, smoothed over 0.25 s (16 frames), and compare. (An unsmoothed per-frame noise spectrum would be cheating: no real system knows the exact noise realization.)

In [ ]:
def smooth_over_time(power, frames=16):
    kernel = np.ones(frames) / frames
    return np.apply_along_axis(lambda column: np.convolve(column, kernel, mode='same'), 0, power)

for kind in ('fan', 'street', 'keyboard', 'babble'):
    noisy_k, noise_k = main.mix_at_snr(clean, main.make_noise(kind, len(clean)), 5.0)
    tracked = main.snr_db(clean, main.enhance(noisy_k)) - 5.0
    oracle = main.snr_db(clean, main.enhance(noisy_k, noise_psd=smooth_over_time(np.abs(main.stft(noise_k)) ** 2))) - 5.0
    print(f"{kind:<9} tracked noise {tracked:+5.1f} dB   true noise, smoothed {oracle:+5.1f} dB")